# G'Contest 2026 - Fraud & Anomaly Detection

Notebook này dùng dữ liệu thật trong `Processed_Data/`. Dữ liệu không có nhãn fraud đã xác minh, vì vậy bài làm không tạo synthetic label. Mục tiêu là xây dựng framework phát hiện bất thường, xếp hạng giao dịch cần review và giải thích nguyên nhân theo nghiệp vụ ngân hàng.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from fraud_pipeline import PipelineConfig, run_pipeline, load_reference_tables, aggregate_activity, validate_schema

RAW_DIR = PROJECT_ROOT / 'Processed_Data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
sns.set_theme(style='whitegrid')

## 2. Assignment Interpretation

Theo assignment, fraud track cần kết hợp transaction metadata với digital footprint để:

- xây behavioral baseline cho từng khách hàng,
- phát hiện account takeover và unauthorized transfers,
- mở rộng sang money laundering patterns nếu dữ liệu cho phép,
- xuất được human-readable reason cho từng dự báo.

Do không có confirmed fraud label, notebook coi đây là bài toán unsupervised/risk-ranking, không phải supervised fraud classification.

## 3. Data Overview And Schema Check

In [ ]:
tables = load_reference_tables(RAW_DIR)
row_counts = {name: len(df) for name, df in tables.items()}
row_counts

In [ ]:
# Activity table is large, so the production pipeline aggregates it by customer-date.
# Run this cell if you want a fresh schema report before the full pipeline.
activity_daily, activity_no_threshold = aggregate_activity(RAW_DIR, chunksize=1_000_000)
schema_report = validate_schema(RAW_DIR, tables, activity_daily)
pd.DataFrame(schema_report).T

## 4. Cause-First Fraud Hypotheses

BGK có xu hướng đánh giá cao việc phân tích nguyên nhân trước. Framework gom tín hiệu thành ba nhánh:

1. **Account takeover / identity compromise**: thiết bị mới, IP mới, hoạt động đêm, người thụ hưởng mới, late-stage activity.
2. **Unauthorized transfer / capital outflow**: chuyển khoản ra ngoài, số tiền lệch baseline, daily burst, dòng tiền ra lớn so với CASA.
3. **AML network / mule-account pattern**: IP/device dùng chung, beneficiary nhận tiền từ nhiều khách hàng, giao dịch số tròn giá trị cao.

`ACTIVITY_NO` được dùng như thứ tự hành động: số lớn hơn là bước sau hơn. Pipeline lấy top 10% `ACTIVITY_NO` làm late-stage activity dựa trên phân phối thật.

In [ ]:
pd.DataFrame({
    'root_cause_branch': [
        'Account takeover / identity compromise',
        'Unauthorized transfer / capital outflow',
        'AML network / mule-account pattern',
    ],
    'signals': [
        'New device/IP, night access, new beneficiary, late-stage activity',
        'Outside-bank transfer, high amount vs customer baseline, daily burst, cash-out vs CASA',
        'Shared device/IP/beneficiary, round high-value transfers, repeated external transfers',
    ],
    'business_action': [
        'Step-up authentication and account verification',
        'Manual review / temporary hold for high-risk transfer',
        'AML escalation and network investigation',
    ],
})

## 5. Run Full Pipeline

In [ ]:
config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    figures_dir=OUTPUT_DIR / 'figures',
    report_dir=PROJECT_ROOT / 'report',
)
metrics = run_pipeline(config)
metrics['review_queue']

## 6. Risk Outputs

In [ ]:
risk = pd.read_csv(OUTPUT_DIR / 'transaction_risk_scores.csv')
customers = pd.read_csv(OUTPUT_DIR / 'customer_risk_summary.csv')
root_cause = pd.read_csv(OUTPUT_DIR / 'root_cause_summary.csv')
risk.head()

In [ ]:
risk['risk_band'].value_counts().reindex(['Low','Medium','High','Critical'])

## 7. Monthly / Quarterly Stability Backtest

In [ ]:
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_stability.csv')
quarterly = pd.read_csv(OUTPUT_DIR / 'quarterly_stability.csv')
monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(monthly['month'], monthly['avg_risk_score'], marker='o', label='Avg risk score')
ax1.set_ylabel('Avg risk score')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['high_critical_rate']*100, marker='s', color='#C46243', label='High/Critical rate')
ax2.set_ylabel('High/Critical rate (%)')
plt.title('Monthly stability backtest on 2019 data')
plt.tight_layout()

## 8. Explainability Examples

In [ ]:
risk[[
    'transaction_row_id', 'CUSTOMER_NUMBER', 'TRANS_DATE', 'TRANS_HOUR', 'TRANS_LV1', 'TRANS_LV2',
    'TRANS_AMOUNT', 'risk_score_0_100', 'risk_band', 'primary_cause_branch',
    'top_reasons', 'recommended_action'
]].head(10)

## 9. SHAP xAI Surrogate

In [ ]:
# Run once after the main pipeline if SHAP files do not exist:
# !python src/xai_shap_engine.py
shap_importance = pd.read_csv(OUTPUT_DIR / 'shap_feature_importance.csv')
shap_local = pd.read_csv(OUTPUT_DIR / 'shap_local_explanations.csv')
shap_importance.head(15)

In [ ]:
shap_local.head(10)

## 10. Evaluation Without Fraud Labels

Vì dữ liệu không có nhãn fraud, không báo precision/recall trên nhãn tự tạo. Evaluation hợp lệ gồm:

- kiểm tra schema và data quality,
- kiểm tra phân phối risk band để phù hợp capacity review,
- kiểm tra các top-risk transaction có reason codes rõ ràng,
- kiểm tra ba nhánh nguyên nhân có output riêng,
- chuẩn bị feedback loop để khi investigator xác nhận case thì đo Precision@K/Recall@K thật.

In [ ]:
with open(OUTPUT_DIR / 'model_metrics.json', encoding='utf-8') as f:
    metrics = json.load(f)
pd.Series(metrics['risk_band_distribution'])

In [ ]:
pd.DataFrame(metrics['top_surrogate_features'].items(), columns=['feature', 'importance']).head(15)

## 11. Report Figures

In [ ]:
from IPython.display import Image, display
for path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 12. Live Demo And Slide Deck

Sau khi chạy pipeline, có thể demo bằng CLI hoặc Streamlit:

```bash
python src/customer_risk_advisor.py --top-critical 3
python src/customer_risk_advisor.py --customer-id <CUSTOMER_NUMBER>
python src/customer_risk_advisor.py --transaction-id <transaction_row_id>
streamlit run src/demo_app.py
```

Tạo slide deck PDF/PPTX:

```bash
python src/build_slide_deck.py
```

Demo trả về risk band, nguyên nhân chính, reason codes, SHAP evidence và recommended action bằng ngôn ngữ dễ hiểu cho risk officer.